# Nov 1 Task Spaceship Titanic Answer Key

This notebook contains the completed code for the Nov 1 Spaceship Titanic task. It recreates the feature engineering and model comparison workflow used to produce a stronger submission.

The workflow creates `TotalSpending`, `CabinDeck`, `CabinSide`, `GroupSize`, and `IsAlone`, then compares logistic regression and Random Forest models.

## 1. Connect Google Drive and load the data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/spaceship_titanic"

In [ ]:
from pathlib import Path
import pandas as pd

data_folder = Path(DATA_DIR)
train = pd.read_csv(data_folder / "train.csv")
test = pd.read_csv(data_folder / "test.csv")
sample_submission = pd.read_csv(data_folder / "sample_submission.csv")

print("Training data loaded:", train.shape)
print("Test data loaded:", test.shape)
print("Sample submission loaded:", sample_submission.shape)
display(train.head())

## 2. Import the machine learning tools

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 3. Create a consistent validation split

In [ ]:
target = "Transported"
y = train[target].astype(int)

train_indices, valid_indices = train_test_split(
    train.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(valid_indices))

## 4. Create engineered features

We create total spending, cabin deck, cabin side, group size, and whether each passenger traveled alone.

In [ ]:
train_engineered = train.copy()
test_engineered = test.copy()

spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

for dataframe in [train_engineered, test_engineered]:
    dataframe["TotalSpending"] = dataframe[spending_columns].sum(
        axis=1,
        min_count=1
    )
    dataframe["CabinDeck"] = dataframe["Cabin"].str.split("/").str[0]
    dataframe["CabinSide"] = dataframe["Cabin"].str.split("/").str[2]
    dataframe["GroupId"] = dataframe["PassengerId"].str.split("_").str[0]

all_passenger_ids = pd.concat([
    train_engineered["PassengerId"],
    test_engineered["PassengerId"]
])
all_group_ids = all_passenger_ids.str.split("_").str[0]
group_id_counts = all_group_ids.value_counts()

for dataframe in [train_engineered, test_engineered]:
    dataframe["GroupSize"] = dataframe["GroupId"].map(group_id_counts)
    dataframe["IsAlone"] = (dataframe["GroupSize"] == 1).astype(int)

new_columns = [
    "TotalSpending", "CabinDeck", "CabinSide",
    "GroupSize", "IsAlone"
]
display(train_engineered[new_columns].head())

## 5. Build the preprocessing and model function

Numerical missing values are filled with the median. Categorical missing values are filled with the most frequent value and then one-hot encoded.

In [ ]:
def build_model(categorical_features, numerical_features, classifier):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

## 6. Choose the input features

In [ ]:
drop_columns = [target, "PassengerId", "Name", "Cabin", "GroupId"]

categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "TotalSpending",
    "GroupSize",
    "IsAlone"
]

X_engineered = train_engineered.drop(columns=drop_columns)
print("Model input columns:", list(X_engineered.columns))

## 7. Train and compare two models

In [ ]:
logistic_model = build_model(
    categorical_features,
    numerical_features,
    LogisticRegression(max_iter=1000)
)

random_forest_model = build_model(
    categorical_features,
    numerical_features,
    RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
)

logistic_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])
logistic_predictions = logistic_model.predict(X_engineered.loc[valid_indices])
logistic_accuracy = accuracy_score(y.loc[valid_indices], logistic_predictions)

random_forest_model.fit(X_engineered.loc[train_indices], y.loc[train_indices])
random_forest_predictions = random_forest_model.predict(X_engineered.loc[valid_indices])
random_forest_accuracy = accuracy_score(y.loc[valid_indices], random_forest_predictions)

model_results = pd.DataFrame({
    "Model": ["Engineered Logistic Regression", "Engineered Random Forest"],
    "Validation Accuracy": [logistic_accuracy, random_forest_accuracy]
}).sort_values("Validation Accuracy", ascending=False).reset_index(drop=True)

display(model_results)

## 8. Select the better model and create a submission

In [ ]:
if logistic_accuracy >= random_forest_accuracy:
    best_model = logistic_model
    best_model_name = "Engineered Logistic Regression"
else:
    best_model = random_forest_model
    best_model_name = "Engineered Random Forest"

best_model.fit(X_engineered, y)

test_drop_columns = [
    column for column in drop_columns
    if column in test_engineered.columns
]
test_input = test_engineered.drop(columns=test_drop_columns)
test_predictions = best_model.predict(test_input).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions
})

submission_path = data_folder / "nov_1_model_submission.csv"
submission.to_csv(submission_path, index=False)

print("Selected model:", best_model_name)
print("Submission saved to:", submission_path)
display(submission.head())

## 9. Check the submission file

In [ ]:
print("Submission shape:", submission.shape)
print("Expected test rows:", len(test))
print("Submission columns:", list(submission.columns))
print("Missing predictions:", submission["Transported"].isna().sum())

assert len(submission) == len(test)
assert list(submission.columns) == ["PassengerId", "Transported"]
assert submission["Transported"].isna().sum() == 0

print("All submission checks passed.")

## 10. Interpretation

The model with the higher validation accuracy was selected for the submission. Validation accuracy is an estimate, so the Kaggle score may be different when Kaggle evaluates the predictions using its hidden labels.

The main lesson is that feature engineering and model comparison can help us improve a baseline, but we should change one component at a time and record the results.